In [1]:
# notebook to generate manifest for alt allele sat mut predictions
# required columns:
# job_id
# job_id_ref
# vcf_path
# enhancer_chrom
# enhancer_id

In [1]:
# import packages
import pandas as pd
import os

In [4]:
# open raw data
allGTEx = pd.read_csv('../processed_data/highPIP_eQTL_mpac_all.tsv.gz', sep = '\t')
allGTEx.loc[:, 'variant_id'] = [('_').join(i.split('_')[0:-1]) for i in allGTEx['variant_id']]
# filter for lead variants and enhancer ids
leadVarEnhancers = allGTEx.filter(['variant_id', 'enhancer_id']).copy().drop_duplicates()

In [5]:
allGTEx.head()

,variant_id,chrom,pos,ref,alt,enhancer_id,lead_tissue,phenotype_id,gene_name,pip,...,sknsh_ref_pred,sknsh_alt_pred,sknsh_skew_pred,k562_tissue_concordant,hepg2_tissue_concordant,sknsh_tissue_concordant,n_gene_associations,n_tissues_significant,cross_tissue_direction_consistent,cross_gene_direction_consistent
0,chr1_906982_C_T,chr1,906982,C,T,EH38E2776603,Cells_Cultured_fibroblasts,ENSG00000230699.2,ENSG00000230699,0.999674,...,0.571255,-0.025078,-0.596333,False,False,False,1,2,True,True
1,chr1_906982_C_T,chr1,906982,C,T,EH38E2776603,Esophagus_Mucosa,ENSG00000230699.2,ENSG00000230699,1.000000,...,0.571255,-0.025078,-0.596333,False,False,False,1,2,True,True
2,chr1_930939_G_A,chr1,930939,G,A,EH38E2776655,Brain_Cortex,ENSG00000187583.11,PLEKHN1,0.965329,...,0.948596,0.945514,-0.003082,False,False,True,1,1,True,True
3,chr1_942951_C_T,chr1,942951,C,T,EH38E2776681,Testis,ENSG00000187634.13,SAMD11,0.952665,...,0.294918,0.251424,-0.043494,False,False,False,1,1,True,True
4,chr1_973946_C_T,chr1,973946,C,T,EH38E2776737,Pituitary,ENSG00000237973.1,MTCO1P12,0.963338,...,0.495122,0.460328,-0.034793,False,False,False,1,1,True,True


In [12]:
# define a function for building VCF
def buildVCF (variantID,
              varSeparator,
              enhancerID):
    vcf = pd.DataFrame({
    '#CHROM' : [variantID.split(varSeparator)[0]],
    'POS' : [int(variantID.split(varSeparator)[1])],
    'ID' : [f'{enhancerID}_{variantID.split('_')[2]}_{variantID.split('_')[3]}'],
    'REF' : [variantID.split(varSeparator)[2]],
    'ALT' : [variantID.split(varSeparator)[-1]],
    'QUAL' : ['.'],
    'FILTER' : ['.'],
    'INFO' : ['.']
    })
    return vcf
# define function for writing vcf to disk
def write_vcf(df, path, meta_lines=("##fileformat=VCFv4.2",)):
    with open(path, "w") as f:
        for line in meta_lines:
            f.write(line + "\n")
        df.to_csv(f, sep="\t", index=False)

In [10]:
# define a function for building manifest
def manifesting (variantIDs, # list of variants for building VCF/IDs
                 varSep, # separator of variant ID
                 enhancerIDs, # list of enhancer IDs, build version with start/stop/etc later but we won't need that now
                 vcfDestination, # path to VCF output
                 finalOut): # path for final manifest output
    # iterate through all lead variants and enhancers to:
    # 1. make manifest file
    # 2. write VCF for building custom genome
    # make lists for building manifest df
    jobID = []
    jobIDref = []
    vcfPath = []
    enhChrom = []
    enhIDs = []
    for varID, enhID in zip(variantIDs,
                            enhancerIDs):
        # build the job id (enhID + ref_alt)
        jobID.append(f'{enhID}_{('_').join(varID.split(varSep))}')
        # build the ref id
        jobIDref.append(f'{enhID}_REF')
        # build the vcf path
        path4VCF = f'{vcfDestination}/{enhID}_{('_').join(varID.split(varSep))}.vcf'
        # build the VCF
        vcf2save = buildVCF(varID,
                            varSep,
                            enhID)
        # save VCF
        write_vcf(vcf2save, path4VCF)
        # bgzip vcf
        os.system(f'bgzip -f {path4VCF}')
        # index vcf
        os.system(f'tabix -p vcf {path4VCF}.gz')
        # update path to vcf
        vcfPath.append(f'{path4VCF}.gz')
        # update chromosome
        enhChrom.append(f'{varID.split(varSep)[0]}')
        # update enhancer ID
        enhIDs.append(enhID)
    # build manifest
    manifest2return = pd.DataFrame({
        'job_id' : jobID,
        'job_id_ref' : jobIDref,
        'vcf_path' : vcfPath,
        'enhancer_chrom' : enhChrom,
        'enhancer_id' : enhIDs
    })
    # save manifest
    manifest2return.to_csv(finalOut, sep = '\t', index = False)

In [11]:
manifesting(
    leadVarEnhancers['variant_id'],
    '_',
    leadVarEnhancers['enhancer_id'],
    '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/analyses/gtex_mechanism_study/processed_data/vcfs4genomes',
    '../processed_data/gtex_phenocopy_alt_sat_mut_v1.tsv'
)